# Interpolation
This notebook runs a linear interpolation of SLA data in the month of Jan 2020.

In [2]:
import pandas as pd
import numpy as np

In [3]:
data_df = pd.read_csv('/home/mhen/geol0069_data/jan20.csv')

In [11]:
#keep data from january

data_df = data_df.loc[data_df['date_string'].str.contains('2020-01', na=False)]

In [12]:
data_df.head()

,x,y,lon,lat,t,z,track,date_string,satellite,lead_mask,dist_along_track,z_track_avg
2573650,-585146.902831,-432100.026835,-53.556094,83.483790,18262.0,NaN,1243,2020-01-01,cs2,0.0,0.000000,0.040157
2573651,-584280.987782,-430861.767792,-53.594113,83.496633,18262.0,-0.034818,1243,2020-01-01,cs2,1.0,1502.990248,0.040157
2573652,-584107.820675,-430614.140368,-53.601734,83.499201,18262.0,-0.018846,1243,2020-01-01,cs2,1.0,1803.558120,0.040157
2573653,-583934.553028,-430366.439101,-53.609361,83.501770,18262.0,NaN,1243,2020-01-01,cs2,0.0,2104.243152,0.040157
2573654,-583588.157115,-429871.156267,-53.624632,83.506906,18262.0,NaN,1243,2020-01-01,cs2,0.0,2705.434872,0.040157


In [5]:
test_track = data_df.loc[data_df['track'] == 1251]
test_track.head()

,x,y,lon,lat,t,z,track,date_string,satellite,lead_mask,dist_along_track,z_track_avg
2588121,1.299330e+06,301319.312827,103.05631,78.034879,18262.0,-0.012229,1251,2020-01-01,cs2,1.0,0.000000,0.060972
2588122,1.299631e+06,301331.522138,103.05390,78.032213,18262.0,NaN,1251,2020-01-01,cs2,0.0,300.933258,0.060972
2588123,1.300835e+06,301381.067972,103.04430,78.021547,18262.0,NaN,1251,2020-01-01,cs2,0.0,1504.737483,0.060972
2588124,1.301437e+06,301405.666626,103.03950,78.016215,18262.0,NaN,1251,2020-01-01,cs2,0.0,2106.543895,0.060972
2588125,1.301738e+06,301417.928656,103.03710,78.013549,18262.0,NaN,1251,2020-01-01,cs2,0.0,2407.450437,0.060972


In [57]:
def linear_interpolation(track_df, train_leads, filter_width_km=100):
    """CPOM-style linear interpolation: box filter smoothing + linear interpolation."""
    obs = np.asarray(track_df['z'].astype(float))
    lead_mask = track_df['lead_mask'].astype(bool)
    leads_idx = lead_mask[lead_mask].index
    
    #only use train_leads for interpolation
    effective_leads = leads_idx & train_leads
    
    # apply 100 km box filter to smooth lead observations
    smoothed_obs = obs.copy()

    x_coords = np.asarray(track_df['x'])
    y_coords = np.asarray(track_df['y'])
    dist_along_track = np.asarray(track_df['dist_along_track'])

    
    if x_coords is not None and y_coords is not None:        
        # For each lead point, smooth with nearby leads within filter_width_km
        for i in effective_leads:
            lead_distances = np.abs(dist_along_track[lead_idx] - dist_along_track[i])
            within_window = lead_distances <= (filter_width_km * 1000 / 2)  # ±50 km
            
            if np.sum(within_window) > 0:
                smoothed_obs[i] = np.mean(obs[lead_idx[within_window]])
    
    # Step 2: Linear interpolation between smoothed lead values
    ssha = np.full_like(obs, np.nan, dtype=float)
    
    for i0, i1 in zip(lead_idx[:-1], lead_idx[1:]):
        v0, v1 = smoothed_obs[i0], smoothed_obs[i1]
        span = i1 - i0
        if span <= 0:
            ssha[i0] = v0
            continue
        ii = np.arange(i0, i1 + 1)
        ssha[ii] = v0 + (v1 - v0) * (ii - i0) / span
    
    # Flat extrapolation beyond the first/last training lead
    first_idx, last_idx = lead_idx[0], lead_idx[-1]
    ssha[:first_idx] = smoothed_obs[first_idx]
    ssha[last_idx:] = smoothed_obs[last_idx]

    return ssha

def random_8020_holdout(data_df, holdout_fraction=0.2, seed=42):
    #create random
    rng = np.random.default_rng(seed)

    #separate into tracks and iterate over
    tracks = data_df['track'].unique()
    results_df = pd.DataFrame()
    for i, track in enumerate(tracks):
            track_df = data_df.loc[data_df['track'] == track]
            
            #locate leads
            leads_df = track_df.loc[track_df['lead_mask']==1.0]
            #skip if too few
            if len(leads_df) < 5:
                        print(f"{prefix} Insufficient leads ({len(leads_df)}) for random holdout")
                        continue
            #create test (holdout) indices of length 1 or of holdout fraction, whichever is larger
            n_test = max(1, int(len(leads_df) * holdout_fraction))
            test_lead_idx = rng.choice(leads_df.index, size=n_test, replace=False)
            train_lead_idx = np.array(leads_df.index.difference(test_lead_idx))

            #run linear interpolation
            print(f'Running lin interp for track {track}, {i+1} of {len(tracks)}...')
            ssha = linear_interpolation(track_df, train_lead_idx)
            track_df['f_linear'] = ssha

            results_df = pd.concat([results_df, track_df])

    return results_df


In [58]:
random_8020_holdout(data_df)

Running lin interp for track 1243, 1 of 4638...
(array([   0,    1,    2, ..., 1287, 1288, 1289]),)


/tmp/ipykernel_878201/4294321827.py:8: FutureWarning: Index.__and__ operating as a set operation is deprecated, in the future this will be a logical operation matching Series.__and__.  Use index.intersection(other) instead.
  effective_leads = leads_idx & train_leads


TypeError: only integer scalar arrays can be converted to a scalar index

In [ ]:
def linear_interpolation(self, obs, lead_mask, train_leads=None, x_coords=None, y_coords=None, filter_width_km=100):
        """CPOM-style linear interpolation: box filter smoothing + linear interpolation."""
        obs = np.asarray(obs, dtype=float)
        lead_mask = np.asarray(lead_mask, dtype=bool)
        
        # If train_leads specified, only use those for interpolation
        if train_leads is not None:
            effective_leads = lead_mask & train_leads
        else:
            effective_leads = lead_mask
            
        valid_leads = effective_leads & np.isfinite(obs) & (obs >= -0.5) & (obs <= 0.5)
        lead_idx = np.where(valid_leads)[0]
        
        if lead_idx.size < 2:
            return np.full_like(obs, np.nan, dtype=float)
        
        # Step 1: Apply 100 km box filter to smooth lead observations
        smoothed_obs = obs.copy()
        
        if x_coords is not None and y_coords is not None:
            # Compute along-track distance for proper spatial filtering
            D_m = np.concatenate([[0], np.cumsum(np.sqrt(np.diff(x_coords)**2 + np.diff(y_coords)**2))])
            
            # For each lead point, smooth with nearby leads within filter_width_km
            for i in lead_idx:
                lead_distances = np.abs(D_m[lead_idx] - D_m[i])
                within_window = lead_distances <= (filter_width_km * 1000 / 2)  # ±50 km
                
                if np.sum(within_window) > 0:
                    smoothed_obs[i] = np.mean(obs[lead_idx[within_window]])
        
        # Step 2: Linear interpolation between smoothed lead values
        ssha = np.full_like(obs, np.nan, dtype=float)
        
        for i0, i1 in zip(lead_idx[:-1], lead_idx[1:]):
            v0, v1 = smoothed_obs[i0], smoothed_obs[i1]
            span = i1 - i0
            if span <= 0:
                ssha[i0] = v0
                continue
            ii = np.arange(i0, i1 + 1)
            ssha[ii] = v0 + (v1 - v0) * (ii - i0) / span
        
        # Flat extrapolation beyond the first/last training lead
        first_idx, last_idx = lead_idx[0], lead_idx[-1]
        ssha[:first_idx] = smoothed_obs[first_idx]
        ssha[last_idx:] = smoothed_obs[last_idx]

        return ssha

In [ ]:
def random_holdout_test(self, holdout_fraction=0.2, seed=42, mode='random'):
        """Randomly hold out a fraction of leads and compare methods."""
        label = "Random" if mode == 'random' else "End-cap"
        print(f"\n{label} holdout test ({int(holdout_fraction*100)}% of leads)...")
        rng = np.random.default_rng(seed)
        results = {}
        for stream_name in ['plrm', 'sar']:
            prefix = f"[{stream_name.upper()}]"
            stream_data = self.data_plrm if stream_name == 'plrm' else self.data_sar
            lead_indices = np.where(stream_data['lead_mask'])[0]
            if len(lead_indices) < 5:
                print(f"{prefix} Insufficient leads ({len(lead_indices)}) for random holdout")
                results[stream_name] = None
                continue
            n_test = max(1, int(len(lead_indices) * holdout_fraction))
            if mode == 'random':
                test_lead_idx = rng.choice(lead_indices, size=n_test, replace=False)
                train_lead_idx = np.setdiff1d(lead_indices, test_lead_idx, assume_unique=True)
            else:
                k = n_test
                if 2 * k >= len(lead_indices):
                    k = max(1, (len(lead_indices) - 2) // 2)
                test_lead_idx = np.concatenate([lead_indices[:k], lead_indices[-k:]])
                train_lead_idx = lead_indices[k:-k]
            train_mask = np.zeros(len(stream_data['h']), dtype=bool)
            test_mask = np.zeros(len(stream_data['h']), dtype=bool)
            train_mask[train_lead_idx] = True
            test_mask[test_lead_idx] = True
            # Compute distances before interpolation
            D_m = np.concatenate([[0], np.cumsum(np.sqrt(np.diff(stream_data['x'])**2 + np.diff(stream_data['y'])**2))])
            lead_D = D_m[lead_indices]
            train_D = D_m[train_lead_idx]
            test_D = D_m[test_lead_idx]
            if train_D.size > 0 and test_D.size > 0:
                nearest_dists_km = np.min(np.abs(test_D[:, None] - train_D[None, :]), axis=1) / 1000.0
            else:
                nearest_dists_km = np.array([])
            linear_ssha = self.linear_interpolation(stream_data['h'], stream_data['lead_mask'], train_mask,
                                                     x_coords=stream_data['x'], y_coords=stream_data['y'])
            gpsat_ssha = self.gpsat_along_track(stream_data, f'{stream_name}_rand_holdout', train_mask)
            gpsat_variance_full = getattr(self, '_last_gpsat_variance', None)
            
            linear_preds = linear_ssha[test_mask]
            gpsat_preds = gpsat_ssha[test_mask]
            targets = stream_data['h'][test_mask]
            
            # Extract GPSat variance at test locations
            if gpsat_variance_full is not None:
                gpsat_variance_test = gpsat_variance_full[test_mask]
            else:
                gpsat_variance_test = None
            print(f"{prefix} Test NaNs — Linear: {np.sum(~np.isfinite(linear_preds))}, GPSat: {np.sum(~np.isfinite(gpsat_preds))}")
            both_valid = (np.isfinite(linear_preds) & np.isfinite(gpsat_preds) & np.isfinite(targets))
            if np.sum(both_valid) == 0:
                print(f"{prefix} FAILED: No valid predictions from both methods on random holdout")
                results[stream_name] = None
                continue
            lin_res = linear_preds[both_valid] - targets[both_valid]
            gp_res = gpsat_preds[both_valid] - targets[both_valid]
            def _r2(a, b):
                return np.corrcoef(a, b)[0,1]**2 if len(a) > 1 else np.nan
            
            # Calculate GPSat uncertainty at test leads for calibration check
            if gpsat_variance_test is not None:
                gpsat_std_test = np.sqrt(gpsat_variance_test[both_valid])
                avg_gpsat_uncertainty = float(np.mean(gpsat_std_test))
            else:
                avg_gpsat_uncertainty = None
            
            # Distance and lead spacing stats
            dist_mean = float(np.nanmean(nearest_dists_km)) if nearest_dists_km.size else np.nan
            dist_median = float(np.nanmedian(nearest_dists_km)) if nearest_dists_km.size else np.nan
            dist_max = float(np.nanmax(nearest_dists_km)) if nearest_dists_km.size else np.nan
            dist_min = float(np.nanmin(nearest_dists_km)) if nearest_dists_km.size else np.nan
            spacing_km = np.diff(lead_D) / 1000.0 if lead_D.size > 1 else np.array([])
            sp_med = float(np.nanmedian(spacing_km)) if spacing_km.size else np.nan
            sp_p90 = float(np.nanpercentile(spacing_km, 90)) if spacing_km.size else np.nan
            sp_max = float(np.nanmax(spacing_km)) if spacing_km.size else np.nan
            results[stream_name] = {
                'n_predictions': int(np.sum(both_valid)),
                'n_train': int(len(train_lead_idx)),
                'n_test': int(len(test_lead_idx)),
                'nearest_train_km': {
                    'mean': dist_mean, 'median': dist_median, 'max': dist_max, 'min': dist_min
                },
                'lead_spacing_km': {
                    'median': sp_med, 'p90': sp_p90, 'max': sp_max
                },
                'linear': {
                    'rmse': float(np.sqrt(np.mean(lin_res**2))),
                    'bias': float(np.mean(lin_res)),
                    'mae': float(np.mean(np.abs(lin_res))),
                    'r2': float(_r2(linear_preds[both_valid], targets[both_valid]))
                },
                'gpsat_along': {
                    'rmse': float(np.sqrt(np.mean(gp_res**2))),
                    'bias': float(np.mean(gp_res)),
                    'mae': float(np.mean(np.abs(gp_res))),
                    'r2': float(_r2(gpsat_preds[both_valid], targets[both_valid])),
                    'avg_uncertainty': avg_gpsat_uncertainty
                },
                # Store plot data for lead-based tests too
                'plot_data': {
                    'linear_preds': linear_preds[both_valid].copy(),
                    'gpsat_preds': gpsat_preds[both_valid].copy(),
                    'targets': targets[both_valid].copy(),
                    'nearest_dists_km': nearest_dists_km[both_valid] if nearest_dists_km.size > 0 else np.array([]),
                    'linear_residuals': lin_res.copy(),
                    'gpsat_residuals': gp_res.copy(),
                    # Store full arrays for along-track visualization
                    'linear_ssha_full': linear_ssha.copy(),
                    'gpsat_ssha_full': gpsat_ssha.copy(),
                    'gpsat_variance_full': getattr(self, '_last_gpsat_variance', None),  # Will be set during gpsat run
                    'train_mask': train_mask.copy(),
                    'test_lead_idx': test_lead_idx.copy()
                }
            }
            print(f"{prefix} {label} holdout: N_train={len(train_lead_idx)}, N_test={len(test_lead_idx)}")
            if nearest_dists_km.size:
                print(f"{prefix} Nearest-train distance (km): mean={dist_mean:.1f}, median={dist_median:.1f}, max={dist_max:.1f}")
            print(f"{prefix} Linear  — RMSE={results[stream_name]['linear']['rmse']:.4f}m, R²={results[stream_name]['linear']['r2']:.3f}")
            print(f"{prefix} GPSat   — RMSE={results[stream_name]['gpsat_along']['rmse']:.4f}m, R²={results[stream_name]['gpsat_along']['r2']:.3f}")
        # Store results with mode-specific attribute name
        if mode == 'random':
            self.random_holdout_results = results
        else:
            self.endcap_holdout_results = results
        # Save random holdout summary
        out_file = 'random_holdout_summary.txt' if mode == 'random' else 'endcap_holdout_summary.txt'
        out_path = self.output_dir / out_file
        with open(out_path, 'w') as f:
            title = "RANDOM HOLDOUT (20%) RESULTS" if mode == 'random' else "END-CAP HOLDOUT RESULTS"
            f.write(title + "\n")
            f.write("="*60 + "\n")
            for stream in ['plrm', 'sar']:
                f.write(f"\n{stream.upper()}\n")
                res = results.get(stream)
                if not res:
                    f.write("  No valid predictions.\n")
                    continue
                f.write(f"  N_train={res['n_train']}, N_test={res['n_test']}, N_valid={res['n_predictions']}\n")
                nd = res.get('nearest_train_km', {})
                if nd:
                    f.write(f"  Nearest-train distance (km): mean={nd.get('mean', float('nan')):.1f}, median={nd.get('median', float('nan')):.1f}, max={nd.get('max', float('nan')):.1f}, min={nd.get('min', float('nan')):.1f}\n")
                ls = res.get('lead_spacing_km', {})
                if ls:
                    f.write(f"  Lead spacing (km): median={ls.get('median', float('nan')):.1f}, p90={ls.get('p90', float('nan')):.1f}, max={ls.get('max', float('nan')):.1f}\n")
                f.write(f"  Linear: RMSE={res['linear']['rmse']:.4f}m, Bias={res['linear']['bias']:.4f}m, MAE={res['linear']['mae']:.4f}m, R²={res['linear']['r2']:.3f}\n")
                gpsat_line = f"  GPSat : RMSE={res['gpsat_along']['rmse']:.4f}m, Bias={res['gpsat_along']['bias']:.4f}m, MAE={res['gpsat_along']['mae']:.4f}m, R²={res['gpsat_along']['r2']:.3f}"
                if res['gpsat_along'].get('avg_uncertainty') is not None:
                    gpsat_line += f", AvgUnc={res['gpsat_along']['avg_uncertainty']:.4f}m"
                f.write(gpsat_line + "\n")
        print(f"{label} holdout test completed. Summary written to {out_file}")
        return results